# 🧬 Protein Structure Visualizer
Upload a **PDB / mmCIF** file or fetch from **RCSB PDB** by accession code.

▶️ **Run all cells**: `Runtime → Run all`

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys
for pkg in ['py3Dmol', 'biopython', 'ipywidgets']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Done.')

In [ ]:
# Cell 2 — Imports
import py3Dmol
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests, io, re
from Bio.PDB import MMCIFParser, PDBParser
import warnings; warnings.filterwarnings('ignore')
print('Imports OK.')

In [ ]:
# Cell 3 — Helpers

CHAIN_PALETTES = {
    'Rainbow': ['#e74c3c','#e67e22','#f1c40f','#2ecc71','#1abc9c','#3498db','#9b59b6','#e91e63'],
    'Neon':    ['#ff073a','#ff6b35','#ffdd00','#39ff14','#00fff5','#bc13fe','#ff00c8','#ff9e00'],
    'Pastel':  ['#ff9aa2','#ffb7b2','#ffdac1','#e2f0cb','#b5ead7','#c7ceea','#f8c8d4','#a8d8ea'],
    'Ocean':   ['#05668d','#028090','#00b4d8','#90e0ef','#48cae4','#0077b6','#023e8a','#ade8f4'],
    'Fire':    ['#ff0000','#ff4500','#ff7f00','#ffaa00','#ffd000','#ffee00','#ff6b6b','#c0392b'],
    'Sunset':  ['#f72585','#b5179e','#7209b7','#560bad','#480ca8','#3a0ca3','#3f37c9','#4361ee'],
}

COLOUR_MODES = [
    'Rainbow (per chain)', 'Neon (per chain)', 'Pastel (per chain)',
    'Ocean (per chain)', 'Fire (per chain)', 'Sunset (per chain)',
    'Secondary structure', 'Spectrum (N to C)', 'B-factor', 'Element (CPK)',
]
REPR_MODES = ['Cartoon', 'Stick', 'Sphere', 'Line', 'Surface']
BG_OPTIONS = [('Black','black'),('White','white'),('Dark Navy','0x0d1117'),
              ('Midnight','0x1a1a2e'),('Light Grey','0xf5f5f5')]

WATER_RESN = ['HOH','WAT','H2O','DOD']

def fetch_pdb(pdb_id):
    pid = pdb_id.strip().upper()
    r = requests.get(f'https://files.rcsb.org/download/{pid}.pdb', timeout=20)
    if r.status_code != 200:
        r = requests.get(f'https://files.rcsb.org/download/{pid}.cif', timeout=20)
        r.raise_for_status()
        return r.text, 'cif', pid
    return r.text, 'pdb', pid

def parse_structure(content, fmt):
    handle = io.StringIO(content)
    parser = PDBParser(QUIET=True) if fmt in ('pdb','ent') else MMCIFParser(QUIET=True)
    s = parser.get_structure('x', handle)
    chains   = list(s.get_chains())
    residues = list(s.get_residues())
    atoms    = list(s.get_atoms())
    aa  = [r for r in residues if r.get_id()[0] == ' ']
    het = [r for r in residues if r.get_id()[0] != ' ']
    chain_ids = [c.id for c in chains]
    stats = {'chains': len(chains), 'ids': ', '.join(chain_ids),
             'aa': len(aa), 'atoms': len(atoms), 'het': len(het)}
    return chain_ids, stats

def make_viewer(content, fmt, repr_mode, colour_mode, bg, show_lig, show_wat, chain_ids):
    rep = repr_mode.lower()  # cartoon | stick | sphere | line | surface
    mol_fmt = 'pdb' if fmt in ('pdb','ent') else 'cif'
    palette_key = colour_mode.split(' (')[0]

    v = py3Dmol.view(width=840, height=560)
    v.setBackgroundColor(bg)
    v.addModel(content, mol_fmt)

    # ── 1. Start clean: hide everything ─────────────────────────────────
    v.setStyle({}, {})

    # ── 2. Style the protein backbone (hetflag=False = standard residues) 
    #       Use setStyle for the whole protein first, then addStyle per
    #       chain to override colours. This keeps a single representation.
    prot_sel = {'hetflag': False}

    if colour_mode == 'Secondary structure':
        if rep == 'surface':
            v.addSurface(py3Dmol.VDW, {'opacity': 0.85, 'colorscheme': 'ssJmol'}, prot_sel)
        else:
            v.setStyle(prot_sel, {rep: {'color': 'ss'}})

    elif colour_mode == 'Spectrum (N to C)':
        if rep == 'surface':
            v.addSurface(py3Dmol.VDW, {'opacity': 0.85, 'colorscheme': 'roygb'}, prot_sel)
        else:
            v.setStyle(prot_sel, {rep: {'color': 'spectrum'}})

    elif colour_mode == 'B-factor':
        if rep == 'surface':
            v.addSurface(py3Dmol.VDW, {'opacity': 0.85, 'colorscheme': 'bwr'}, prot_sel)
        else:
            v.setStyle(prot_sel, {rep: {'colorscheme': 'bwr'}})

    elif colour_mode == 'Element (CPK)':
        if rep == 'surface':
            v.addSurface(py3Dmol.VDW, {'opacity': 0.85, 'colorscheme': 'Jmol'}, prot_sel)
        else:
            v.setStyle(prot_sel, {rep: {'colorscheme': 'Jmol'}})

    else:
        # Per-chain palette:
        # First set the entire protein to the representation with a neutral color,
        # then override per chain with addStyle.
        colours = CHAIN_PALETTES.get(palette_key, CHAIN_PALETTES['Rainbow'])
        if rep == 'surface':
            for i, cid in enumerate(chain_ids):
                v.addSurface(py3Dmol.VDW,
                              {'opacity': 0.85, 'color': colours[i % len(colours)]},
                              {'chain': cid, 'hetflag': False})
        else:
            # Set the whole protein once (gives it the representation)
            v.setStyle(prot_sel, {rep: {'color': colours[0]}})
            # Then override each chain's colour
            for i, cid in enumerate(chain_ids):
                v.setStyle({'chain': cid, 'hetflag': False},
                            {rep: {'color': colours[i % len(colours)]}})

    # ── 3. Ligands: select HETATM atoms, then hide water on top ─────────
    #    Strategy: apply ligand style to ALL hetatm, then selectively
    #    hide water with an empty style override. No 'invert' needed.
    if show_lig:
        v.addStyle({'hetflag': True},
                    {'stick':  {'colorscheme': 'Jmol', 'radius': 0.2},
                     'sphere': {'colorscheme': 'Jmol', 'radius': 0.4}})

    # ── 4. Water: always hide unless requested ───────────────────────────
    if show_wat:
        # Show water as small blue spheres (overrides ligand stick if shown)
        v.setStyle({'resn': WATER_RESN},
                    {'sphere': {'color': '#00bfff', 'radius': 0.2}})
    else:
        # Explicitly wipe water style so it stays hidden
        v.setStyle({'resn': WATER_RESN}, {})

    v.zoomTo()
    return v

print('Helpers ready.')

In [ ]:
# Cell 4 — Interactive UI

_s = {'content': None, 'fmt': None, 'name': 'N/A', 'src': 'N/A', 'chain_ids': []}

upload_w = widgets.FileUpload(
    accept='.pdb,.ent,.cif,.mmcif', multiple=False,
    description='Upload file', layout=widgets.Layout(width='200px'))

pdb_w = widgets.Text(
    placeholder='e.g. 6LU7', description='RCSB ID:',
    layout=widgets.Layout(width='200px'))

fetch_w = widgets.Button(
    description='Fetch from RCSB', button_style='info',
    icon='download', layout=widgets.Layout(width='170px'))

repr_w = widgets.Dropdown(
    options=REPR_MODES, value='Cartoon',
    description='Representation:', layout=widgets.Layout(width='270px'))

colour_w = widgets.Dropdown(
    options=COLOUR_MODES, value='Rainbow (per chain)',
    description='Colour:', layout=widgets.Layout(width='270px'))

bg_w = widgets.Dropdown(
    options=BG_OPTIONS, value='black',
    description='Background:', layout=widgets.Layout(width='230px'))

lig_w = widgets.Checkbox(value=True,  description='Show ligands', indent=False)
wat_w = widgets.Checkbox(value=False, description='Show water',   indent=False)

render_w   = widgets.Button(
    description='Render / Update', button_style='success',
    icon='play', layout=widgets.Layout(width='180px', height='40px'))

status_out = widgets.Output()
viewer_out = widgets.Output()

def on_upload(change):
    if not upload_w.value:
        return
    info = list(upload_w.value.values())[0]
    name = info['metadata']['name']
    fmt  = name.rsplit('.', 1)[-1].lower()
    text = info['content'].tobytes().decode('utf-8', errors='replace')
    _s.update({'content': text, 'fmt': fmt, 'name': name, 'src': 'Local upload'})
    with status_out:
        clear_output()
        print(f'Loaded: {name} ({len(text):,} chars) — click Render.')

upload_w.observe(on_upload, names='value')

def on_fetch(_):
    pid = pdb_w.value.strip()
    if not re.match(r'^[A-Za-z0-9]{4}$', pid):
        with status_out:
            clear_output(); print('Enter a valid 4-character PDB ID (e.g. 6LU7).')
        return
    with status_out:
        clear_output(); print(f'Fetching {pid.upper()} ...')
    try:
        content, fmt, name = fetch_pdb(pid)
        _s.update({'content': content, 'fmt': fmt, 'name': name, 'src': f'RCSB ({name})'})
        with status_out:
            clear_output(); print(f'Fetched {name} ({len(content):,} chars) — click Render.')
    except Exception as e:
        with status_out:
            clear_output(); print(f'Error: {e}')

fetch_w.on_click(on_fetch)

def on_render(_):
    if _s['content'] is None:
        with status_out:
            clear_output(); print('No structure loaded yet.')
        return
    with status_out:
        clear_output(); print('Rendering ...')
    try:
        chain_ids, stats = parse_structure(_s['content'], _s['fmt'])
        _s['chain_ids'] = chain_ids
        v = make_viewer(_s['content'], _s['fmt'], repr_w.value, colour_w.value,
                         bg_w.value, lig_w.value, wat_w.value, chain_ids)
        info_html = (
            f"<div style='font-family:monospace;background:#0d1117;color:#58a6ff;"
            f"border:1px solid #30363d;border-radius:8px;padding:10px 16px;"
            f"margin-bottom:8px;line-height:1.9;'>"
            f"<b style='color:#e6edf3'>{_s['name']}</b>"
            f"<span style='color:#8b949e;float:right'>{_s['src']}</span><br>"
            f"<span style='color:#8b949e'>Chains </span>{stats['chains']} ({stats['ids']}) "
            f"<span style='color:#8b949e'>| Residues </span>{stats['aa']} "
            f"<span style='color:#8b949e'>| Atoms </span>{stats['atoms']} "
            f"<span style='color:#8b949e'>| Hetatoms </span>{stats['het']}</div>"
        )
        with viewer_out:
            clear_output(wait=True)
            display(widgets.HTML(info_html))
            v.show()
        with status_out:
            clear_output()
            print('Done!  Drag = rotate  |  Scroll = zoom  |  Right-drag = pan')
    except Exception as e:
        import traceback
        with status_out:
            clear_output(); print(f'Render error: {e}'); traceback.print_exc()

render_w.on_click(on_render)

display(widgets.VBox([
    widgets.HTML('<h3 style="font-family:monospace;color:#58a6ff;margin:4px 0 12px">'
                 'Protein Structure Visualizer</h3>'),
    widgets.HBox([
        widgets.VBox([
            widgets.HTML('<b>Load structure</b>'),
            upload_w,
            widgets.HTML('<i style="color:grey;font-size:12px">— or fetch from RCSB —</i>'),
            widgets.HBox([pdb_w, fetch_w]),
        ], layout=widgets.Layout(margin='0 40px 0 0')),
        widgets.VBox([
            widgets.HTML('<b>Appearance</b>'),
            repr_w, colour_w, bg_w,
            widgets.HBox([lig_w, wat_w]),
        ]),
    ]),
    widgets.HTML('<hr style="border-color:#ccc;margin:12px 0">'),
    render_w,
    status_out,
    viewer_out,
]))

---
### Colour modes
| Mode | Description |
|------|-------------|
| Rainbow / Neon / Pastel / Ocean / Fire / Sunset | Vivid explicit hex colours per chain |
| Secondary structure | Helix = red, Sheet = yellow, Loop = green |
| Spectrum (N to C) | Blue N-terminus to red C-terminus |
| B-factor | Blue (rigid) to red (flexible) |
| Element (CPK) | Standard atom-type colours |

### Controls
| Action | Mouse |
|--------|-------|
| Rotate | Left-click + drag |
| Zoom | Scroll wheel |
| Pan | Right-click + drag |

### Quick test IDs
- **6LU7** — SARS-CoV-2 Mpro + N3 inhibitor
- **1HHO** — Haemoglobin (4 chains)
- **1CRN** — Crambin (tiny, fast)
- **4HHB** — Deoxyhaemoglobin